# FRP++2_Ex: A-B-D matrices
#### **(by Norbert Blanco)**

This notebook computes the A-B-D matrices of a composite laminate with UD plies.

Converted from the original MATLAB Live Script (`FRPpp2_Ex0501.m`) to Python. It runs in Jupyter and in Google Colab without any additional setup (only `numpy` and `pandas`, both preinstalled in Colab).

## Initialization

Import the packages used throughout the notebook.

In [ ]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=False)


## Definition of material, LSS and applied loads

#### Definition of material properties

- The material is assumed to be transversely isotropic.
- The plies are assumed to be working under plane-stress condition. So, only 4 elastic constants are required: E11 (MPa), E22 (MPa), nu12, G12 (MPa).
- CFRP T300/5208 is defined as material 1. Additional materials can be defined following the same structure `Mat[mat_num] = [E11, E22, nu12, G12]`, or by updating the material properties.

In [ ]:
# Mat[mat_num] = [E11, E22, nu12, G12]
Mat = {
    1: [181000, 10300, 0.28, 7170],
}


#### Definition of the Laminate Stacking Sequence

- The LSS is defined through material (previously defined), orientation (in degrees) and thickness (in mm).
- The first layer is assumed to be the one on the bottom, according to the Z direction.
- Define as many layers as required following this structure: `L[layer_num] = [mat_num, orientation, thickness]`.
- If the laminate is symmetric, set `LSS_symm = True` and only define the first half of the LSS. For LSS with odd symmetry, define the central ply with half of the thickness.

In [ ]:
LSS_symm = False

# L[layer_num] = [mat_num, orientation (deg), thickness (mm)]
L = np.array([
    [1,  0, 2.0],
    [1, 45, 1.0],
])


## Calculations

First it is necessary to complete the LSS in case a symmetric LSS is defined:

In [ ]:
n_lam = L.shape[0]

if LSS_symm:
    L = np.vstack([L, L[::-1]])

n_lam = L.shape[0]
L


#### Calculation and assembly of stiffness, compliance and transformation matrices

The plane-stress compliance matrix S is calculated for each ply according to the definition for a transversely isotropic material. The plane-stress stiffness matrix Q is obtained by inverting S:

$$
S = \begin{bmatrix}
1/E_{11} & -\nu_{12}/E_{11} & 0 \\
-\nu_{12}/E_{11} & 1/E_{22} & 0 \\
0 & 0 & 1/G_{12}
\end{bmatrix}
$$

In [ ]:
S = np.zeros((3, 3, n_lam))
Q = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    i_mat = int(L[i, 0])
    E11, E22, nu12, G12 = Mat[i_mat]

    S[0, 0, i] = 1 / E11
    S[0, 1, i] = -nu12 / E11
    S[1, 0, i] = S[0, 1, i]
    S[1, 1, i] = 1 / E22
    S[2, 2, i] = 1 / G12

    Q[:, :, i] = np.linalg.inv(S[:, :, i])


The transformation matrix T is calculated for each ply according to the definition. The transformation matrix for strain, `Tg`, is also obtained according to the definition.

In [ ]:
T = np.zeros((3, 3, n_lam))
Tg = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    theta = L[i, 1] * np.pi / 180
    m = np.cos(theta)
    n = np.sin(theta)

    T[0, 0, i] = m**2
    T[0, 1, i] = n**2
    T[0, 2, i] = 2 * m * n
    T[1, 0, i] = n**2
    T[1, 1, i] = m**2
    T[1, 2, i] = -2 * m * n
    T[2, 0, i] = -m * n
    T[2, 1, i] = m * n
    T[2, 2, i] = m**2 - n**2

    Tg[:, :, i] = np.linalg.inv(T[:, :, i]).T


The transformed compliance and stiffness matrices are obtained for each ply:

In [ ]:
#@title { display-mode: "form" }
Sb = np.zeros((3, 3, n_lam))
Qb = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    Sb[:, :, i] = np.linalg.inv(Tg[:, :, i]) @ S[:, :, i] @ T[:, :, i]
    Qb[:, :, i] = np.linalg.inv(Sb[:, :, i])

for i in range(n_lam):
    print(f"Qb(:,:,{i + 1}) =")
    print(Qb[:, :, i])
    print()
    print(f"Sb(:,:,{i + 1}) =")
    print(Sb[:, :, i])
    print()


### Calculation of the laminate's constitutive equation

Before determining matrices A, B and D, it is necessary to determine the total thickness of the laminate and the coordinates of each ply surface from the laminate's mid-plane:

In [ ]:
TH = L[:, 2].sum()
th = TH / 2

Z = np.zeros(n_lam + 1)
Z[0] = -th
running = 0.0
for i in range(n_lam):
    running += L[i, 2]
    Z[i + 1] = running - th

Z


#### Calculation of matrices A, B and D

Matrices A, B and D are calculated taking into account the bottom and top coordinates of the layers and the corresponding stiffness matrices:

$$
A_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} (Z_{i+1} - Z_i) \qquad
B_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} \frac{Z_{i+1}^2 - Z_i^2}{2} \qquad
D_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} \frac{Z_{i+1}^3 - Z_i^3}{3}
$$

In [ ]:
#@title { display-mode: "form" }
A = np.zeros((3, 3))
B = np.zeros((3, 3))
D = np.zeros((3, 3))

for i in range(n_lam):
    A += Qb[:, :, i] * (Z[i + 1] - Z[i])
    B += Qb[:, :, i] * (Z[i + 1]**2 - Z[i]**2) / 2
    D += Qb[:, :, i] * (Z[i + 1]**3 - Z[i]**3) / 3

ABD = np.block([[A, B], [B, D]])

abd_labels = ['A11', 'A12', 'A16', 'B11', 'B12', 'B16']
ABD_df = pd.DataFrame(ABD, index=abd_labels, columns=abd_labels)
pd.set_option('display.float_format', lambda v: f'{v:,.0f}'.replace(',', ' '))

print("ABD =")
display(ABD_df)


For convenience, matrices A, B and D are also shown separately below.

In [ ]:
#@title { display-mode: "form" }
xyk_labels = ['1', '2', '6']
A_df = pd.DataFrame(A, index=xyk_labels, columns=xyk_labels)
B_df = pd.DataFrame(B, index=xyk_labels, columns=xyk_labels)
D_df = pd.DataFrame(D, index=xyk_labels, columns=xyk_labels)

print("A =")
display(A_df)
print("[N/mm]")
print()
print("B =")
display(B_df)
print("[N]")
print()
print("D =")
display(D_df)
print("[N·mm]")
